# 3.11 Zaman Serileri

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/11-working-with-time-series.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Working with Time Series

Pandas başlangıçta finansal modelleme bağlamında geliştirildi; beklenebileceği gibi tarihler, saatler ve zaman indeksli verilerle çalışmak için kapsamlı bir araç seti içerir. Tarih ve saat verisi birkaç türde gelir:

Bu bölüm Pandas'ta bu tarih/saat veri türlerinin her biriyle nasıl çalışılacağını tanıtır. Python veya Pandas'taki zaman serisi araçlarının eksiksiz bir rehberi değil; kullanıcı olarak zaman serileriyle nasıl yaklaşmanız gerektiğine dair geniş bir bakış sunar. Python'daki tarih ve saat araçlarının kısa bir tartışmasıyla başlayacağız; ardından Pandas'ın sunduğu araçlara geçeceğiz. Daha derin kaynakları listeledikten sonra Pandas'ta zaman serisi verisiyle çalışmanın kısa örneklerini gözden geçireceğiz.

## Python'da Tarihler ve Saatler

Python dünyasında tarih, saat, fark ve zaman aralıklarının birkaç temsili vardır. Pandas'ın zaman serisi araçları veri bilimi uygulamalarında genelde en kullanışlı olsa da diğer Python araçlarıyla ilişkisini görmek faydalıdır.

### Yerleşik Python Tarih/Saat: datetime ve dateutil

Python'da tarih ve saatle çalışmak için temel nesneler yerleşik datetime modülündedir. Üçüncü taraf dateutil modülüyle tarihler üzerinde bir dizi kullanışlı işlemi hızlıca yapabilirsiniz. Örneğin datetime tipiyle elle tarih oluşturabilirsiniz:


In [ ]:
# datetime_ornek.py
from datetime import datetime
datetime(year=2021, month=7, day=4)



Esnek biçimde biçimlendirilmiş bir dize tarihini ayrıştırmak için dateutil modülünü kullanabilirsiniz:


In [ ]:
# dateutil_parse.py
from dateutil import parser
date = parser.parse("4th of July, 2021")
date



datetime nesneniz olduktan sonra haftanın gününü yazdırmak gibi işlemler yapabilirsiniz:


In [ ]:
# strftime_gun.py
date.strftime('%A')



Burada tarihleri yazdırmak için standart dize biçim kodlarından birini ('%A') kullandık; bunlar Python datetime dokümantasyonundaki strftime bölümünde açıklanır. Diğer yararlı tarih yardımcılarının dokümantasyonu dateutil çevrimiçi dokümantasyonunda bulunur. Bilinmesi gereken ilgili paket pytz; zaman serisi verisinin en baş ağrıtan unsuru olan saat dilimleriyle çalışma araçlarını içerir.

datetime ve dateutil'in gücü esneklik ve kolay sözdizimindedir: ilgilendiğiniz neredeyse her işlemi bu nesneler ve yerleşik yöntemleriyle kolayca yapabilirsiniz. Zayıf kaldıkları yer büyük tarih/saat dizileriyle çalışmak istediğinizdedir: Python sayısal değişken listeleri NumPy tarzı tipli sayısal dizilere göre verimsiz olduğu gibi Python datetime nesne listeleri de kodlanmış tarih dizilerine göre verimsizdir.

### Zaman Tipli Diziler: NumPy datetime64


In [ ]:
# np_datetime64.py
import numpy as np
date = np.array('2021-07-04', dtype=np.datetime64)
date



Tarihleri bu biçimde elde ettikten sonra vektörize işlemleri hızlıca yapabiliriz:


In [ ]:
# datetime64_arange.py
date + np.arange(12)



NumPy datetime64 dizilerindeki tekdüze tip sayesinde bu tür işlem, diziler büyüdükçe özellikle Python datetime nesneleriyle doğrudan çalışmaktan çok daha hızlı yapılabilir (vektörizasyonu 2.3 Evrensel Fonksiyonlar bölümünde tanıtmıştık).

datetime64 ve ilgili timedelta64 nesnelerinin bir ayrıntısı temel zaman birimi üzerine kurulu olmalarıdır. datetime64 64 bit hassasiyetle sınırlı olduğundan kodlanabilir zaman aralığı bu temel birimin $2^{64}$ katıdır. Yani datetime64 zaman çözünürlüğü ile maksimum zaman aralığı arasında ödünleşim dayatır.

Örneğin 1 nanosaniye çözünürlük istiyorsanız yalnızca $2^{64}$ nanosaniye, yani yaklaşık 600 yıl kodlayabilirsiniz. NumPy istenen birimi girdiden çıkarır; örneğin gün tabanlı datetime:


In [ ]:
# datetime64_gun.py
np.datetime64('2021-07-04')



İşte dakika tabanlı bir datetime:


In [ ]:
# datetime64_dakika.py
np.datetime64('2021-07-04 12:00')



İstenen temel birimi birçok biçim kodundan biriyle zorlayabilirsiniz; burada nanosaniye tabanlı zaman zorluyoruz:


In [ ]:
# datetime64_ns.py
np.datetime64('2021-07-04 12:59:59.50', 'ns')



Aşağıdaki tablo NumPy datetime64 dokümantasyonundan alınmış olup kullanılabilir biçim kodlarını ve kodlayabildikleri göreli/mutlak zaman aralıklarını listeler:

Gerçek dünyada gördüğümüz veri türleri için kullanışlı bir varsayılan datetime64[ns]'dir; modern tarihlerin geniş bir aralığını uygun ince çözünürlükle kodlayabilir.

Son olarak datetime64 yerleşik Python datetime tipinin bazı eksikliklerini giderse de datetime ve özellikle dateutil'in sağladığı birçok kullanışlı yöntem ve fonksiyondan yoksundur. Daha fazla bilgi NumPy datetime64 dokümantasyonunda.

### Pandas'ta Tarih ve Saat: İki Dünyanın En İyisi

Pandas az önce tartışılan tüm araçların üzerine inşa ederek Timestamp nesnesi sunar; datetime ve dateutil'in kullanım kolaylığını numpy.datetime64'ün verimli depolama ve vektörize arayüzüyle birleştirir. Bu Timestamp nesnelerinden Pandas, Series veya DataFrame'de veriyi indekslemek için DatetimeIndex oluşturabilir.

Örneğin daha önceki gösterimi Pandas araçlarıyla tekrarlayabiliriz. Esnek biçimli dize tarihini ayrıştırıp gün adını biçim kodlarıyla yazdırabiliriz:


In [ ]:
# pd_to_datetime.py
import pandas as pd
date = pd.to_datetime("4th of July, 2021")
date



In [ ]:
# pd_strftime.py
date.strftime('%A')



Ayrıca aynı nesne üzerinde NumPy tarzı vektörize işlemler yapabiliriz:


In [ ]:
# pd_timedelta.py
date + pd.to_timedelta(np.arange(12), 'D')



> **Not**
>

Sonraki bölümde Pandas'ın sunduğu araçlarla zaman serisi verisini manipüle etmeye daha yakından bakacağız.

## Pandas Zaman Serisi: Zamana Göre İndeksleme

Pandas zaman serisi araçları veriyi zaman damgalarıyla indekslemeye başladığınızda gerçekten faydalı olur. Örneğin zaman indeksli veri içeren bir Series oluşturabiliriz:


In [ ]:
# datetime_index_series.py
index = pd.DatetimeIndex(['2020-07-04', '2020-08-04',
                          '2021-07-04', '2021-08-04'])
data = pd.Series([0, 1, 2, 3], index=index)
data



Veriyi bir Series'te tuttuğumuza göre önceki bölümlerdeki Series indeksleme kalıplarının herhangi birini, tarihe dönüştürülebilen değerler geçirerek kullanabiliriz:


In [ ]:
# series_date_slice.py
data['2020-07-04':'2021-07-04']



Ek özel yalnızca-tarih indeksleme işlemleri vardır; örneğin bir yıl geçirerek o yıldaki tüm verinin dilimini almak:


In [ ]:
# series_year_slice.py
data['2021']



### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Zaman indeksli Series ile dilimleme deneyin:
      
        import pandas as pd
idx = pd.DatetimeIndex([&quot;2020-01-01&quot;, &quot;2020-06-01&quot;, &quot;2021-01-01&quot;])
s = pd.Series([10, 20, 30], index=idx)
print(s[&quot;2020&quot;:&quot;2020-12-31&quot;])

Daha sonra tarihleri indeks olarak kullanmanın kolaylığının ek örneklerini göreceğiz. Önce mevcut zaman serisi veri yapılarına daha yakından bakalım.

## Pandas Zaman Serisi Veri Yapıları

Bu bölüm zaman serisi verisiyle çalışmak için temel Pandas yapılarını tanıtır:


In [ ]:
# to_datetime_coklu.py
dates = pd.to_datetime([datetime(2021, 7, 3), '4th of July, 2021',
                       '2021-Jul-6', '07-07-2021', '20210708'])
dates



Herhangi bir DatetimeIndex, frekans kodu eklenerek to_period ile PeriodIndex'e dönüştürülebilir; burada günlük frekans için 'D' kullanıyoruz:


In [ ]:
# to_period.py
dates.to_period('D')



Örneğin bir tarihten diğeri çıkarıldığında TimedeltaIndex oluşur:


In [ ]:
# dates_fark.py
dates - dates[0]



## Düzenli Diziler: pd.date_range

Düzenli tarih dizileri oluşturmayı kolaylaştırmak için Pandas pd.date_range (zaman damgaları), pd.period_range (dönemler) ve pd.timedelta_range (zaman farkları) sunar. Python range ve NumPy np.arange başlangıç, bitiş ve isteğe bağlı adım alıp dizi döndürür. Benzer şekilde pd.date_range başlangıç/bitiş tarihi ve isteğe bağlı frekans kodu ile düzenli tarih dizisi oluşturur:


In [ ]:
# date_range.py
pd.date_range('2015-07-03', '2015-07-10')



Alternatif olarak bitiş yerine başlangıç noktası ve dönem sayısı belirtilebilir:


In [ ]:
# date_range_periods.py
pd.date_range('2015-07-03', periods=8)



freq argümanı (varsayılan D) aralığı değiştirir. Örneğin saatlik zaman damgaları dizisi:


In [ ]:
# date_range_hourly.py
pd.date_range('2015-07-03', periods=8, freq='H')



freq argümanı (varsayılan D) aralığı değiştirir. Örneğin saatlik zaman damgaları dizisi:


In [ ]:
# period_range.py
pd.period_range('2015-07', periods=8, freq='M')



Düzenli Period veya Timedelta dizileri için pd.period_range ve pd.timedelta_range kullanışlıdır. Aylık dönemler:


In [ ]:
# timedelta_range.py
pd.timedelta_range(0, periods=6, freq='H')



### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      pd.date_range ile haftalık tarih dizisi oluşturun:
      
        import pandas as pd
dr = pd.date_range(&quot;2024-01-01&quot;, periods=5, freq=&quot;W-MON&quot;)
print(dr)

Saat saat artan süre dizisi:

## Frekanslar ve Ofsetler

Pandas zaman serisi araçlarının temelinde frekans veya tarih ofseti kavramı yatar. Ana kodların özeti aşağıdadır; önceki bölümlerdeki D (gün) ve H (saat) gibi istediğiniz frekans aralığını belirtmek için kullanılır:

Aylık, çeyreklik ve yıllık frekansların hepsi belirtilen dönemin sonunda işaretlenir. Herhangi birine S eki eklemek bunları dönemin başında işaretler:

Ayrıca çeyreklik veya yıllık kodların ayını üç harfli ay koduyla, haftalık frekansın bölünme gününü üç harfli gün koduyla değiştirebilirsiniz (Q-JAN, W-MON vb.). Kodlar sayılarla birleştirilerek başka frekanslar da belirtilebilir; örneğin 2 saat 30 dakika:


In [ ]:
# timedelta_2h30t.py
pd.timedelta_range(0, periods=6, freq="2H30T")



Tüm bu kısa kodlar pd.tseries.offsets modülündeki Pandas zaman serisi ofset örneklerine referans verir:


In [ ]:
# bday_offset.py
from pandas.tseries.offsets import BDay
pd.date_range('2015-07-01', periods=6, freq=BDay())



Frekans ve ofset kullanımının daha fazla tartışması için Pandas dokümantasyonundaki DateOffset bölümüne bakın.

## Yeniden Örnekleme, Kaydırma ve Pencereleme

Tarih/saat indeksleri sezgisel veri organizasyonu sağlar. Pandas ek olarak yeniden örnekleme, kaydırma ve pencereleme sunar. Hisse fiyat örneği için pandas-datareader ile S&P 500 verisi yüklenir:


In [ ]:
# sp500_datareader.py
from pandas_datareader import data

sp500 = data.DataReader('^GSPC', start='2018', end='2022',
                        data_source='yahoo')
sp500.head()



Basitlik için yalnızca kapanış fiyatını kullanacağız:


In [ ]:
# sp500_close.py
sp500 = sp500['Close']



Normal Matplotlib kurulum kodundan sonra plot yöntemiyle görselleştirebiliriz (bkz. kitap Bölüm 4):


In [ ]:
# sp500_plot.py
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
sp500.plot();



### Frekans Yeniden Örnekleme ve Dönüştürme


In [ ]:
# resample_asfreq.py
sp500.plot(alpha=0.5, style='-')
sp500.resample('BA').mean().plot(style=':')
sp500.asfreq('BA').plot(style='--');
plt.legend(['input', 'resample', 'asfreq'],
           loc='upper left');



### Frekans Yeniden Örnekleme ve Dönüştürme

Yukarı örneklemede resample ve asfreq büyük ölçüde eşdeğerdir. Varsayılan olarak boş noktalar NA ile doldurulur; 3.4 Eksik Veri bölümündeki gibi asfreq method ile doldurma yapılabilir:


In [ ]:
# asfreq_fill.py
fig, ax = plt.subplots(2, sharex=True)
data = sp500.iloc[:20]

data.asfreq('D').plot(ax=ax[0], marker='o')

data.asfreq('D', method='bfill').plot(ax=ax[1], style='-o')
data.asfreq('D', method='ffill').plot(ax=ax[1], style='--o')
ax[1].legend(["back-fill", "forward-fill"]);



S&P 500 kapanış verisini aşağı örneklemede ikisinin döndürdüğünü karşılaştıralım. İş yılı sonunda yeniden örnekliyoruz:

### Zaman Kaydırma


In [ ]:
# shift_roi.py
sp500 = sp500.asfreq('D', method='pad')

ROI = 100 * (sp500.shift(-365) - sp500) / sp500
ROI.plot()
plt.ylabel('% Return on Investment after 1 year');



Yukarı örneklemede resample ve asfreq büyük ölçüde eşdeğerdir; resample'ın çok daha fazla seçeneği vardır. Her iki yöntemin varsayılanı yukarı örneklenen noktaları boş bırakmaktır (NA ile doldurulur). 3.4 Eksik Veri bölümündeki pd.fillna gibi asfreq değerlerin nasıl doldurulacağını belirten method argümanı kabul eder. İş günü verisini günlük frekansta (hafta sonları dahil) yeniden örnekliyoruz:

### Kayan Pencereleme

Kayan istatistikler rolling özniteliğiyle hesaplanır (3.8 Agregasyon ve Gruplama):

Örneğin bir yıllık merkezli kayan ortalama ve medyan:


In [ ]:
# rolling_mean_median.py
rolling = sp500.rolling(365, center=True)

data = pd.DataFrame({'input': sp500,
                     'one-year rolling_mean': rolling.mean(),
                     'one-year rolling_median': rolling.median()})
ax = data.plot(style=['-', '--', ':'])
ax.lines[0].set_alpha(0.3)



> **Not**
>

### Zaman Kaydırma

## Daha Fazlasını Nereden Öğrenilir?

Kapsamlı tartışma: Pandas zaman serisi dokümantasyonu ve Wes McKinney Python for Data Analysis. IPython ? ile keşfetmeyi unutmayın.

## Örnek: Seattle Bisiklet Sayımları

Seattle Fremont Köprüsü saatlik bisiklet sayım verisi (2012+). CSV indirme notebook'taki yorum satırlarında:


In [ ]:
# url = ('https://raw.githubusercontent.com/jakevdp/'
#        'bicycle-data/main/FremontBridge.csv')
# !curl -O {url}



CSV'yi Date indeksli ve parse_dates=True ile okuyoruz:


In [ ]:
# read_fremont.py
data = pd.read_csv('FremontBridge.csv', index_col='Date', parse_dates=True)
data.head()



Sütun adlarını kısaltıyoruz:


In [ ]:
# fremont_columns.py
data.columns = ['Total', 'East', 'West']



Özet istatistikler:


In [ ]:
# fremont_describe.py
data.dropna().describe()



### Veriyi Görselleştirme


In [ ]:
# fremont_plot_raw.py
data.plot()
plt.ylabel('Hourly Bicycle Count');



~150.000 saatlik örnek çok yoğundur; haftalık resample ile eğilimleri görelim:


In [ ]:
# fremont_weekly.py
weekly = data.resample('W').sum()
weekly.plot(style=['-', ':', '--'])
plt.ylabel('Weekly bicycle count');



Yaz/kış farkı, hava etkisi ve 2020'de COVID-19'un işe gidip gelme kalıplarına etkisi görülür.


In [ ]:
# fremont_rolling30.py
daily = data.resample('D').sum()
daily.rolling(30, center=True).sum().plot(style=['-', ':', '--'])
plt.ylabel('mean hourly count');



Kayan ortalama penceresinin sert kesimi tırtıklılık yaratır; Gauss win_type ile yumuşatma yapılabilir:


In [ ]:
# fremont_gaussian.py
daily.rolling(50, center=True,
              win_type='gaussian').sum(std=10).plot(style=['-', ':', '--']);



### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Kayan ortalama ile basit zaman serisi yumuşatma:
      
        import pandas as pd
import numpy as np
rng = np.random.default_rng(0)
idx = pd.date_range(&quot;2020-01-01&quot;, periods=100, freq=&quot;D&quot;)
s = pd.Series(rng.random(100).cumsum(), index=idx)
print(s.rolling(7, center=True).mean().tail())

### Veriye Derinlemesine Bakmak


In [ ]:
# fremont_by_time.py
by_time = data.groupby(data.index.time).mean()
hourly_ticks = 4 * 60 * 60 * np.arange(6)
by_time.plot(xticks=hourly_ticks, style=['-', ':', '--']);



Saatlik trafik iki tepelidir (08:00 ve 17:00) — işe gidip gelme kanıtı. Doğu kaldırım sabah, batı akşam daha yoğun.


In [ ]:
# fremont_by_weekday.py
by_weekday = data.groupby(data.index.dayofweek).mean()
by_weekday.index = ['Mon', 'Tues', 'Wed', 'Thurs', 'Fri', 'Sat', 'Sun']
by_weekday.plot(style=['-', ':', '--']);



Haftanın gününe göre groupby:


In [ ]:
# fremont_weekend_group.py
weekend = np.where(data.index.weekday < 5, 'Weekday', 'Weekend')
by_time = data.groupby([weekend, data.index.time]).mean()



Hafta içi/sonu ve saat bazında bileşik groupby; sonuçları Matplotlib ile iki panelde çiziyoruz:


In [ ]:
# fremont_subplots.py
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
by_time.loc['Weekday'].plot(ax=ax[0], title='Weekdays',
                            xticks=hourly_ticks, style=['-', ':', '--'])
by_time.loc['Weekend'].plot(ax=ax[1], title='Weekends',
                            xticks=hourly_ticks, style=['-', ':', '--']);



Hafta içi iki tepeli, hafta sonu tek tepeli kalıp ortaya çıkar. Veri kümesine modelleme bölümünde tekrar döneceğiz.

> **Not**
>
